# ForgeLM baseline training (Kaggle T4)
# 1) Set Accelerator: GPU T4 x2. 2) Add HF_TOKEN secret (write scope) if pushing. 3) Run all.

In [ ]:
import os, shutil
os.chdir("/kaggle/working")
if os.path.exists("/kaggle/working/forge-lm"):
    shutil.rmtree("/kaggle/working/forge-lm")
os.system("git clone https://github.com/manva-niso/ForgeLM.git /kaggle/working/forge-lm")
os.chdir("/kaggle/working/forge-lm")
os.system("git checkout kaggle")
os.system("pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cu121")
os.system("pip install -q --disable-pip-version-check datasets pyyaml tensorboard tqdm huggingface_hub")
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    try:
        os.environ["HF_TOKEN"] = client.get_secret("HF_TOKEN")
    except Exception:
        os.environ["HF_TOKEN"] = client.get_secret("forgelm")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as e:
    print("HF_TOKEN not loaded:", e)
import forger
print("forger import OK")


In [ ]:
!python scripts/kaggle_sft.py --device cuda --steps 3000 --examples 15000 --batch-size 32 --context-length 128 --ckpt models/forgelm-baseline --ckpt-dir /kaggle/working/ckpt_sft --export /kaggle/working/int4_sft --hf-repo Manvaniso/forgelm-sft


In [ ]:
import shutil, os
if os.path.exists("/kaggle/working/ckpt_sft/merged"):
    shutil.make_archive("/kaggle/working/sft_merged", "zip", "/kaggle/working/ckpt_sft/merged")
if os.path.exists("/kaggle/working/int4_sft"):
    shutil.make_archive("/kaggle/working/int4_sft", "zip", "/kaggle/working/int4_sft")
from IPython.display import FileLink
FileLink("/kaggle/working/sft_merged.zip")
FileLink("/kaggle/working/int4_sft.zip")
